# R5 — Supervisor multi-agent

The client's hardest question — asked word for word as "For CLM-20003: which plan applied, is room rent over cap, and what is payable?" — is split by a supervisor between three specialist agents (data, policy, adjudication) and checked by a plain-code verifier that recomputes the payout from the plan rules in claims.raw.plans. Every attempt is traced in MLflow and recorded in agent/traces/r5_attempt1..3.json; each verdict must equal the verified answer in claims.genai.adjudication_verifier (plan at admission, payable true).

In [0]:
# Config — the single place the catalog, models and paths are named for this notebook
import os

CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
REPO_DIR = "/Workspace/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1/Gen AI - Paper 5 - Scenario Based Assignment"

# The client's question, word for word
QUESTION = "For CLM-20003: which plan applied, is room rent over cap, and what is payable?"
CLAIM_ID = "CLM-20003"

spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  chat_model={CHAT_MODEL}")
print(f"question: {QUESTION}")

In [0]:
# The four R1 Unity Catalog tools, described for the LLM and executed against claims.genai.
# Each specialist receives only the tools it needs; run_tool is shared and execute_tool records
# every call (name, args, raw rows) so the supervisor can build the verdict from tool records,
# never from an LLM's paraphrase.
import json

TOOL_DEFS = {
  "lookup_claim": {"type": "function", "function": {
      "name": "lookup_claim",
      "description": "Look up the stored facts of one claim: status, claim type, admission and discharge dates, billed and approved amounts, rejection code, plan at admission, room rent per day and cap, policy tenure, days since policy start, network flag. Returns facts exactly as stored; does not decide a payout.",
      "parameters": {"type": "object", "properties": {"claim_id": {"type": "string", "description": "Claim id, e.g. CLM-10042"}}, "required": ["claim_id"]}}},
  "plan_at": {"type": "function", "function": {
      "name": "plan_at",
      "description": "Return the plan terms (plan code, family, room rent cap, co-pay pct, PED and initial waiting periods, validity dates) a member held on a given date - the policy version valid on that date, never the member's current plan.",
      "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Member id, e.g. MEM-0437"}, "as_of_date": {"type": "string", "description": "Date as YYYY-MM-DD"}}, "required": ["member_id", "as_of_date"]}}},
  "search_policy": {"type": "function", "function": {
      "name": "search_policy",
      "description": "Search the Saffron Shield policy binder (hybrid keyword plus vector). Returns at most 5 sections with chunk_id, doc_name, heading and text, best first. Use for any question about what the policy says.",
      "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Natural-language policy question"}}, "required": ["query"]}}},
  "compute_payable": {"type": "function", "function": {
      "name": "compute_payable",
      "description": "Deterministically decide a claim's payout by the plan rules. Returns a JSON string with room_rent_over_cap, waiting_period_ok, payable, copay_pct, reason_code and payable_amount. The only source of a payout figure - never compute one yourself.",
      "parameters": {"type": "object", "properties": {
          "plan_code": {"type": "string"},
          "is_pre_existing": {"type": "boolean"},
          "policy_tenure_months": {"type": "integer"},
          "days_since_policy_start": {"type": "integer"},
          "is_network_at_admission": {"type": "boolean"},
          "room_rent_per_day": {"type": "integer"},
          "billed_amount": {"type": "number"}},
        "required": ["plan_code", "is_pre_existing", "policy_tenure_months", "days_since_policy_start", "is_network_at_admission", "room_rent_per_day", "billed_amount"]}}},
}

def _q(s: str) -> str:
    return str(s).replace("'", "''")

TOOL_CALLS = []  # calls made by the current specialist run: [{tool, args, rows}]

def run_tool(name: str, args: dict) -> str:
    if name == "lookup_claim":
        rows = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{_q(args['claim_id'])}')").collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "plan_at":
        rows = spark.sql(f"SELECT * FROM claims.genai.plan_at('{_q(args['member_id'])}', DATE'{_q(args['as_of_date'])}')").collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "search_policy":
        rows = spark.sql(f"SELECT chunk_id, doc_name, heading, text FROM claims.genai.search_policy('{_q(args['query'])}')").collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "compute_payable":
        row = spark.sql(f"""
          SELECT claims.genai.compute_payable(
            '{_q(args['plan_code'])}', {args['is_pre_existing']}, {args['policy_tenure_months']},
            {args['days_since_policy_start']}, {args['is_network_at_admission']},
            {args['room_rent_per_day']}, {args['billed_amount']}) AS payout
        """).collect()[0]
        result = row.payout
    else:
        result = json.dumps({"error": f"unknown tool {name}"})
    return result

def execute_tool(name: str, args: dict) -> str:
    result = run_tool(name, args)
    parsed = json.loads(result)
    rows = parsed if isinstance(parsed, list) else [parsed]
    TOOL_CALLS.append({"tool": name, "args": args, "rows": rows})
    return result

print(f"{len(TOOL_DEFS)} tools registered: {sorted(TOOL_DEFS)}")

In [0]:
# Chat call (temperature 0) and the three specialists: data (lookup_claim + plan_at),
# policy (search_policy), adjudication (compute_payable). Each returns its tool records
# ({tool, args, rows}) and its own final answer; the supervisor trusts the records.
import mlflow
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
AUTH = w.config.authenticate()["Authorization"]          # Bearer token, never printed or saved
HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

def chat(messages, tools):
    body = {"messages": messages, "tools": tools, "temperature": 0, "max_tokens": 1024}
    r = requests.post(
        f"{HOST}/serving-endpoints/{CHAT_MODEL}/invocations",
        headers={"Authorization": AUTH, "Content-Type": "application/json"},
        json=body,
    )
    r.raise_for_status()
    return r.json()

SYSTEM_DATA = """You are the data specialist of the Saffron Shield claims supervisor.
You have two tools: lookup_claim (the stored facts of one claim) and plan_at (the plan terms a member held on a given date).
For a claim question: first call lookup_claim with the claim id; then call plan_at with the member_id and admission_date from that row, so the plan you report is the plan valid on the admission date, never the member's current plan.
Then answer with the claim facts and plan terms exactly as the tools returned them. Never compute a payout and never invent a fact."""

SYSTEM_POLICY = """You are the policy specialist of the Saffron Shield claims supervisor.
You have one tool: search_policy, which returns the best-matching sections of the Saffron Shield policy binder.
Search for the sections the question needs - for anything about room rent, search for room rent capping and the proportionate deduction rule.
Answer from the returned sections and cite every section you used by its chunk_id. Never compute a payout."""

def adjudication_system(facts_json: str) -> str:
    return """You are the adjudication specialist of the Saffron Shield claims supervisor.
You must never decide a payout yourself: the only source of a payout is the compute_payable tool.
Here are the verified facts from the data specialist:
""" + facts_json + """
Call compute_payable with exactly these values (types as given), then return the payout JSON it returns, unchanged."""

def specialist(kind: str, question: str, facts_json: str = None, max_steps: int = 4):
    global TOOL_CALLS
    if kind == "data":
        tools, system = [TOOL_DEFS["lookup_claim"], TOOL_DEFS["plan_at"]], SYSTEM_DATA
    elif kind == "policy":
        tools, system = [TOOL_DEFS["search_policy"]], SYSTEM_POLICY
    elif kind == "adjudication":
        tools, system = [TOOL_DEFS["compute_payable"]], adjudication_system(facts_json)
    else:
        raise ValueError(kind)
    TOOL_CALLS = []
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        msg = chat(messages, tools)["choices"][0]["message"]
        tcs = msg.get("tool_calls") or []
        if not tcs:
            return {"tool": [c["tool"] for c in TOOL_CALLS],
                    "data": TOOL_CALLS,
                    "answer": msg.get("content") or ""}
        messages.append(msg)
        for tc in tcs:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            result = execute_tool(name, args)
            messages.append({"role": "tool", "tool_call_id": tc["id"], "content": result})
    raise RuntimeError(f"{kind} specialist did not settle within max_steps")

print("specialists ready: data, policy, adjudication")

In [0]:
# Verifier - plain code, no LLM. Recomputes the payout from the plan rules stored in
# claims.raw.plans (room rent cap, co-pay, PED and initial waiting periods) and checks the
# adjudication verdict field by field. No LLM may decide a payout alone.
import json

def recompute_payout(facts: dict) -> dict:
    plan = spark.sql(
        f"SELECT * FROM claims.raw.plans WHERE plan_code = '{_q(facts['plan_code'])}'"
    ).collect()[0].asDict()
    cap = int(plan["room_rent_cap_per_day"])
    copay = int(plan["copay_pct"])
    ped_months = int(plan["ped_waiting_months"])
    initial_days = int(plan["initial_waiting_days"])
    rent = int(facts["room_rent_per_day"])
    billed = float(facts["billed_amount"])

    over_cap = rent > cap
    waiting_ok = (int(facts["days_since_policy_start"]) >= initial_days) and (
        (not facts["is_pre_existing"]) or int(facts["policy_tenure_months"]) >= ped_months
    )
    deduction = round((1 - cap / rent) * 100, 1) if over_cap else 0.0
    payable = waiting_ok
    amount = round(billed * (1 - deduction / 100) * (1 - copay / 100), 2) if payable else 0.0
    if not waiting_ok:
        reason = "PED_WAITING" if facts["is_pre_existing"] else "INITIAL_WAITING"
    elif over_cap:
        reason = "PAYABLE_WITH_DEDUCTION"
    else:
        reason = "PAYABLE"
    return {"room_rent_over_cap": over_cap, "waiting_period_ok": waiting_ok,
            "payable": payable, "room_rent_deduction_pct": deduction,
            "copay_pct": copay, "reason_code": reason, "payable_amount": amount}

def verifier_check(verdict: dict, facts: dict) -> bool:
    rc = recompute_payout(facts)
    fields = ["room_rent_over_cap", "waiting_period_ok", "payable",
              "room_rent_deduction_pct", "copay_pct", "reason_code", "payable_amount"]
    def eq(f):
        a, b = verdict.get(f), rc[f]
        return abs(a - b) < 0.01 if isinstance(b, float) else a == b
    agree = all(eq(f) for f in fields)
    if not agree:
        print("verifier mismatch:", {f: [verdict.get(f), rc[f]] for f in fields if not eq(f)})
    return agree

# self-test on the known CLM-20003 numbers before the supervisor relies on it
_t = recompute_payout({"plan_code": "SILVER_IND", "room_rent_per_day": 5000,
                       "billed_amount": 240000.0, "is_pre_existing": False,
                       "policy_tenure_months": 19, "days_since_policy_start": 598})
assert _t == {"room_rent_over_cap": True, "waiting_period_ok": True, "payable": True,
              "room_rent_deduction_pct": 40.0, "copay_pct": 0,
              "reason_code": "PAYABLE_WITH_DEDUCTION", "payable_amount": 144000.0}, _t
print("verifier ready (self-test on CLM-20003 numbers passed)")

In [0]:
# Supervisor: routes the question data -> policy -> adjudication -> verifier, builds the
# verdict from the specialists' tool records, and has the verifier (plain code) check it.
# Three attempts are recorded in full in agent/traces/r5_attempt1..3.json; all are kept.
import json
import os
import re

@mlflow.trace(name="saffron_shield_supervisor", span_type="AGENT")
def supervisor_run(question: str):
    route = []

    # 1. data specialist: claim facts + plan at admission
    data = specialist("data", question)
    route.append("data")
    lookup_calls = [c for c in data["data"] if c["tool"] == "lookup_claim"]
    plan_calls = [c for c in data["data"] if c["tool"] == "plan_at"]
    assert lookup_calls and plan_calls, "data specialist must call both lookup_claim and plan_at"
    claim = lookup_calls[0]["rows"][0]
    plan = plan_calls[0]["rows"][0]
    assert claim["claim_id"] == re.search(r"CLM-\d+", question).group(), "wrong claim looked up"
    assert plan["plan_code"] == claim["plan_code_at_admission"], \
        "plan_at must return the plan valid on the admission date"
    facts = {
        "plan_code": claim["plan_code_at_admission"],
        "is_pre_existing": claim["is_pre_existing"],
        "policy_tenure_months": claim["policy_tenure_months"],
        "days_since_policy_start": claim["days_since_policy_start"],
        "is_network_at_admission": claim["is_network_at_admission"],
        "room_rent_per_day": claim["room_rent_per_day"],
        "billed_amount": claim["billed_amount"],
    }

    # 2. policy specialist: the room-rent rule, cited by chunk id
    policy = specialist("policy", question)
    route.append("policy")
    assert policy["tool"] == ["search_policy"], "policy specialist must call search_policy"

    # 3. adjudication specialist: payout only from compute_payable
    adj = specialist("adjudication", question, facts_json=json.dumps(facts))
    route.append("adjudication")
    payout_calls = [c for c in adj["data"] if c["tool"] == "compute_payable"]
    assert payout_calls, "adjudication specialist must call compute_payable"
    payout = payout_calls[0]["rows"][0]

    # 4. verifier (plain code): recompute and check the verdict
    agree = verifier_check(payout, facts)
    route.append("verifier")

    verdict = {
        "plan_code": payout["plan_code"],
        "payable": payout["payable"],
        "waiting_period_ok": payout["waiting_period_ok"],
        "network_at_admission": payout["network_at_admission"],
        "room_rent_over_cap": payout["room_rent_over_cap"],
        "room_rent_deduction_pct": payout["room_rent_deduction_pct"],
        "copay_pct": payout["copay_pct"],
        "reason_code": payout["reason_code"],
        "payable_amount": payout["payable_amount"],
        "final_answer": (
            f"For {claim['claim_id']} the plan at admission was {payout['plan_code']}; "
            f"room rent of {claim['room_rent_per_day']} per day is over the "
            f"{claim['room_rent_cap_per_day']} per day cap, so a "
            f"{payout['room_rent_deduction_pct']}% proportionate deduction applies; "
            f"the claim is payable and the payable amount is {payout['payable_amount']}."
        ),
        "agree": agree,
    }
    mlflow.get_current_active_span().set_attributes({
        "question": question,
        "route": json.dumps(route),
        "verdict": json.dumps(verdict, default=str),
    })
    return {"question": question, "route": route,
            "specialist": {"data": data, "policy": policy, "adjudication": adj},
            "verdict": verdict, "trace_id": mlflow.get_active_trace_id()}

attempts = []
for n in (1, 2, 3):
    out = supervisor_run(QUESTION)
    rec = {"question": QUESTION, "attempt": n, "trace_id": out["trace_id"],
           "route": out["route"], "specialist": out["specialist"],
           "verdict": out["verdict"]}
    attempts.append(rec)
    v = rec["verdict"]
    print(f"attempt {n}: route={rec['route']} trace={rec['trace_id']}")
    print(f"           plan={v['plan_code']} over_cap={v['room_rent_over_cap']} "
          f"payable={v['payable']} amount={v['payable_amount']} agree={v['agree']}")

paths = []
for rec in attempts:
    p = os.path.join(REPO_DIR, "agent", "traces", f"r5_attempt{rec['attempt']}.json")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w") as f:
        json.dump(rec, f, indent=2)
    with open(p) as f:
        assert json.load(f) == rec, "evidence round-trip failed"
    paths.append(p)
    print(f"wrote {p}")

In [0]:
# Grader-style verification: every attempt file carries the required keys, the question word
# for word, the four steps in order, each specialist's tool records, the cited room-rent chunk,
# and a verdict equal to the verified answer in claims.genai.adjudication_verifier. All three
# attempt traces must exist in MLflow.
import json
import os

from mlflow.tracking import MlflowClient

REQUIRED = {"question", "attempt", "trace_id", "route", "specialist", "verdict"}
VERDICT_KEYS = {"plan_code", "payable", "waiting_period_ok", "network_at_admission",
                "room_rent_over_cap", "room_rent_deduction_pct", "copay_pct",
                "reason_code", "payable_amount", "final_answer", "agree"}
CHUNK = "04_Room_Rent_Capping_and_Proportionate_Deduction.md#proportionate-deduction"
ROUTE = ["data", "policy", "adjudication", "verifier"]

key = spark.sql(
    f"SELECT * FROM claims.genai.adjudication_verifier WHERE claim_id = '{CLAIM_ID}'"
).collect()[0].asDict()
lookup = spark.sql(
    f"SELECT * FROM claims.genai.lookup_claim('{CLAIM_ID}')"
).collect()[0].asDict()
client = MlflowClient()

for n in (1, 2, 3):
    p = os.path.join(REPO_DIR, "agent", "traces", f"r5_attempt{n}.json")
    ev = json.load(open(p))
    assert set(ev) == REQUIRED, f"attempt {n}: unexpected keys {sorted(ev)}"
    assert ev["attempt"] == n, f"attempt {n}: attempt key must equal {n}"
    assert ev["question"] == QUESTION, f"attempt {n}: question not word for word"
    assert ev["route"] == ROUTE, f"attempt {n}: route must be data, policy, adjudication, verifier"

    d, pol, adj = (ev["specialist"][k] for k in ("data", "policy", "adjudication"))
    dtools = [c["tool"] for c in d["data"]]
    assert "lookup_claim" in dtools and "plan_at" in dtools, \
        f"attempt {n}: data specialist records must hold the lookup_claim and plan_at calls"
    assert pol["tool"] == ["search_policy"], f"attempt {n}: policy specialist must call search_policy"
    chunks = [r["chunk_id"] for c in pol["data"] for r in c["rows"]]
    assert CHUNK in chunks, f"attempt {n}: policy agent did not return the room-rent chunk"
    assert adj["tool"] == ["compute_payable"], \
        f"attempt {n}: adjudication specialist must call compute_payable"
    for kind in ("data", "policy", "adjudication"):
        assert ev["specialist"][kind]["answer"].strip(), f"attempt {n}: empty {kind} answer"

    v = ev["verdict"]
    assert VERDICT_KEYS <= set(v), f"attempt {n}: verdict missing {VERDICT_KEYS - set(v)}"
    assert v["plan_code"] == lookup["plan_code_at_admission"] == "SILVER_IND", \
        f"attempt {n}: plan_code must be the plan at admission"
    assert v["payable"] == key["payable"] is True, f"attempt {n}: payable disagrees with verifier key"
    assert v["waiting_period_ok"] == key["waiting_period_ok"], f"attempt {n}: waiting_period_ok mismatch"
    assert v["network_at_admission"] == key["network_at_admission"], f"attempt {n}: network mismatch"
    assert abs(v["room_rent_deduction_pct"] - key["room_rent_deduction_pct"]) < 0.01, \
        f"attempt {n}: deduction mismatch"
    assert v["copay_pct"] == key["copay_pct"], f"attempt {n}: copay mismatch"
    assert v["reason_code"] == key["reason_code"], f"attempt {n}: reason_code mismatch"
    assert abs(v["payable_amount"] - 144000.0) < 0.01, f"attempt {n}: payable_amount mismatch"
    assert v["agree"] is True, f"attempt {n}: verifier did not agree"
    assert v["final_answer"].strip(), f"attempt {n}: empty final_answer"
    client.get_trace(ev["trace_id"])  # raises if the trace does not exist in MLflow
    print(f"attempt {n}: keys, route, specialists, cited chunk, verified answer, trace OK")

print("\nPASS: r5_attempt1/2/3.json - verdict equals the verified answer for CLM-20003")